In [3]:
import gzip
import json

filepath = r'E:\OpenAlex\openalex-snapshot\data\works\updated_date=2019-01-01\part_0000.gz'

with gzip.open(filepath, 'rt', encoding='utf-8') as f:
    # Read just the first line to see the structure
    first_line = f.readline()
    work_data = json.loads(first_line)

# Print the top-level keys
print("Main keys in a Work object:")
print(work_data.keys())

# Print a formatted version of the first record
print("\nExample Record Structure:")
print(json.dumps(work_data, indent=2))

Main keys in a Work object:
dict_keys(['id', 'doi', 'title', 'authorships', 'authors_count', 'institutions', 'institutions_distinct_count', 'countries_distinct_count', 'corresponding_author_ids', 'corresponding_institution_ids', 'publication_date', 'publication_year', 'has_abstract', 'referenced_works_count', 'referenced_works', 'related_works', 'abstract_inverted_index', 'cited_by_api_url', 'cited_by_count', 'counts_by_year', 'fwci', 'citation_normalized_percentile', 'ids', 'indexed_in', 'language', 'biblio', 'primary_topic', 'topics', 'topics_key', 'keywords', 'concepts', 'locations_count', 'locations', 'primary_location', 'best_oa_location', 'has_content', 'sustainable_development_goals', 'grants', 'awards', 'funders', 'open_access', 'type', 'is_paratext', 'is_retracted', 'indexed_in_crossref', 'is_xpac', 'mesh', 'created_date', 'display_name'])

Example Record Structure:
{
  "id": "https://openalex.org/W2905966264",
  "doi": "https://doi.org/10.22287/ag.v1i21.645",
  "title": "A IN

## Find first record where primary_topic and refereced_works are not empty

In [8]:
import gzip
import json
import os

root_dir = r"E:\OpenAlex\openalex-snapshot\data\works"

def find_and_display_match(directory):
    for root, _, files in os.walk(directory):
        # Filter for .gz files and sort them to stay organized
        gz_files = [f for f in files if f.endswith(".gz")]
        
        for file in gz_files:
            file_path = os.path.join(root, file)
            print(f"Searching: {file_path}", end='\r') # Dynamic status update
            
            try:
                with gzip.open(file_path, 'rt', encoding='utf-8') as f:
                    for line in f:
                        work = json.loads(line)
                        
                        # VALIDATION LOGIC:
                        # 1. work.get('primary_topic') ensures it's not None
                        # 2. work.get('referenced_works') ensures the list exists 
                        # 3. len(...) > 0 ensures the list isn't empty []
                        has_topic = work.get('primary_topic') is not None
                        has_refs = len(work.get('referenced_works', [])) > 0
                        
                        if has_topic and has_refs:
                            print(f"\n\n{'='*80}")
                            print(f"SUCCESS: MATCH FOUND IN {file}")
                            print(f"{'='*80}\n")
                            
                            # Pretty-print the entire record
                            print(json.dumps(work, indent=4))
                            return work
                            
            except (json.JSONDecodeError, OSError) as e:
                print(f"\nSkipping error in {file}: {e}")
                continue

    print("\nSearch complete. No record found matching both criteria.")
    return None

# Start search
found_work = find_and_display_match(root_dir)

Searching: E:\OpenAlex\openalex-snapshot\data\works\updated_date=2016-06-24\part_0000.gz

SUCCESS: MATCH FOUND IN part_0000.gz

{
    "id": "https://openalex.org/W2283374966",
    "doi": "https://doi.org/10.2495/str030361",
    "title": "Structural analysis of the main apse vault of St. George of Greeks Cathedral built c.1390 at Famagusta, Cyprus",
    "authorships": [
        {
            "affiliations": [],
            "author": {
                "display_name": "Ata Atun",
                "id": "https://openalex.org/A934223916"
            },
            "author_position": "first",
            "countries": [],
            "raw_author_name": "Ata Atun",
            "is_corresponding": true,
            "raw_affiliation_strings": [],
            "institutions": []
        }
    ],
    "authors_count": 1,
    "institutions": [],
    "institutions_distinct_count": 0,
    "countries_distinct_count": 0,
    "corresponding_author_ids": [],
    "corresponding_institution_ids": [],
    "pub

In [ ]:
import gzip
import json
import os
import pandas as pd
from sindex.core.ids import _norm_doi
import re
from concurrent.futures import ProcessPoolExecutor

def shorten_id(url):
    if not url: return None
    return url.split('/')[-1]

# --- Processing Logic ---

SOURCE_DIR = './data/works'
OUTPUT_DIR = './output_parquet'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def process_single_file(file_path):
    metadata_list = []
    citations_list = []
    file_name = os.path.basename(file_path)
    
    try:
        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
            for line in f:
                w = json.loads(line)
                
                # Extract and Normalize basic fields
                work_id = shorten_id(w.get('id'))
                raw_doi = w.get('doi') or ""
                norm_doi = _norm_doi(raw_doi)
                
                # Date Fallback
                pub_date = w.get('publication_date')
                if not pub_date:
                    pub_date = str(w.get('publication_year')) if w.get('publication_year') else None
                
                # --- Task 1: Clean Metadata ---
                topic = w.get('primary_topic') or {}
                metadata_list.append({
                    'id': work_id,
                    'doi': norm_doi,
                    'pub_date': pub_date,
                    'topic_id': shorten_id(topic.get('id')),
                    'topic_name': topic.get('display_name'),
                    'topic_score': topic.get('score')
                })
                
                # --- Task 2: Citations ---
                refs = w.get('referenced_works', [])
                if refs:
                    for r_url in refs:
                        citations_list.append({
                            'citing_id': work_id,
                            'citing_doi': norm_doi,
                            'citing_pub_date': pub_date,
                            'cited_id': shorten_id(r_url)
                        })
        
        # Save to Parquet
        if metadata_list:
            pd.DataFrame(metadata_list).to_parquet(f"{OUTPUT_DIR}/meta_{file_name}.parquet", index=False)
        if citations_list:
            pd.DataFrame(citations_list).to_parquet(f"{OUTPUT_DIR}/cite_{file_name}.parquet", index=False)
            
        return f"Finished {file_name}"
    except Exception as e:
        return f"Error in {file_name}: {e}"

if __name__ == "__main__":
    all_files = [os.path.join(r, f) for r, _, fs in os.walk(SOURCE_DIR) for f in fs if f.endswith('.gz')]
    with ProcessPoolExecutor() as executor:
        list(executor.map(process_single_file, all_files))